In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# Lightweight Cue-On Theta Peak TFR Notebook (HPC + ACC)

This notebook auto-selects cue-on theta-peak electrodes and plots trial-mean TFRs for all available:
- HPC electrodes
- ACC electrodes


In [ ]:
import math
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 300
pd.set_option('display.max_colwidth', 140)


In [ ]:
# ---- Config ----
base_dir = Path('/Volumes/T7 Shield')
event = 'cue_on'
condition = 'Day1'
theta_band = (4.0, 8.0)
event_window = (0.0, 1.0)
selection_metric = 'theta_p90'
plot_all_hpc = True
plot_all_acc = True
freq_plot_range = (2.0, 200.0)
plot_cmap = 'viridis'
plot_interpolation = 'bicubic'
freq_res = 5
time_res = 250

analysis_root = Path('/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/analysis_outputs/cue_on_theta_peak_tfr_hpc_acc')
figure_root = analysis_root / 'figures'
channel_figure_root = figure_root / 'by_channel'
analysis_root.mkdir(parents=True, exist_ok=True)
figure_root.mkdir(parents=True, exist_ok=True)
channel_figure_root.mkdir(parents=True, exist_ok=True)

selection_csv = analysis_root / 'cue_on_theta_peak_selected_electrodes.csv'
missing_csv = analysis_root / 'cue_on_theta_peak_missing_items.csv'
hpc_grid_png = figure_root / 'cue_on_theta_peak_tfr_grid_hpc.pdf'
acc_grid_png = figure_root / 'cue_on_theta_peak_tfr_grid_acc.pdf'

print('analysis_root:', analysis_root)
print('event:', event, '| condition:', condition)
print('theta_band:', theta_band, '| event_window:', event_window)
print('selection_metric:', selection_metric)
assert plot_cmap == 'viridis'
assert plot_interpolation == 'bicubic'
assert freq_res == 5


In [ ]:
# ---- Rank electrodes from lightweight precomputed per-channel dfs ----
ephys_root = base_dir / 'projects/guLab/Salman/EphysAnalyses'

def discover_subjects(root: Path):
    subjects = []
    for p in sorted(root.glob('*')):
        if not p.is_dir():
            continue
        s = p.name
        if not (s.startswith('MS') or s.startswith('UI')):
            continue
        dfs_dir = p / 'scratch' / 'TFR' / event / 'dfs'
        tfr_file = p / 'scratch' / 'TFR' / f'{event}-tfr.h5'
        if dfs_dir.exists() and tfr_file.exists():
            subjects.append(s)
    return subjects

subjects = discover_subjects(ephys_root)
print('discovered subjects:', len(subjects))

rank_rows = []
missing_rows = []

for subject in subjects:
    dfs_dir = ephys_root / subject / 'scratch' / 'TFR' / event / 'dfs'
    csv_paths = sorted(dfs_dir.glob('*_time_averaged_df_time0.0-1.0.csv'))
    if len(csv_paths) == 0:
        missing_rows.append({'subject': subject, 'item_type': 'dfs', 'detail': 'no_time0.0-1.0_csv'})
        continue

    for csv_path in csv_paths:
        try:
            df = pd.read_csv(csv_path, usecols=['region', 'channel', 'fband', 'tfr'])
        except Exception as exc:
            missing_rows.append({'subject': subject, 'item_type': 'dfs_read_error', 'detail': f'{csv_path.name}: {exc}'})
            continue

        df = df[(df['fband'] == 'high_theta') & (df['region'].isin(['HPC', 'ACC']))].copy()
        if len(df) == 0:
            continue

        region = str(df['region'].iloc[0])
        channel = str(df['channel'].iloc[0])
        tfr_values = pd.to_numeric(df['tfr'], errors='coerce').dropna().to_numpy(dtype=np.float64)
        if tfr_values.size == 0:
            continue

        rank_rows.append({
            'subject': subject,
            'region': region,
            'channel': channel,
            'theta_mean': float(np.nanmean(tfr_values)),
            'theta_median': float(np.nanmedian(tfr_values)),
            'theta_p90': float(np.nanpercentile(tfr_values, 90)),
            'theta_p95': float(np.nanpercentile(tfr_values, 95)),
            'theta_max': float(np.nanmax(tfr_values)),
            'n_rows': int(tfr_values.size),
            'source_csv': str(csv_path)
        })

ranking_df = pd.DataFrame(rank_rows)
if len(ranking_df) == 0:
    raise ValueError('No ranking rows were produced from precomputed cue_on dfs files.')

selection_parts = []
for region in ['HPC', 'ACC']:
    reg_df = ranking_df[ranking_df['region'] == region].copy()
    reg_df = reg_df.sort_values([selection_metric, 'theta_p95', 'theta_mean'], ascending=False).reset_index(drop=True)
    selected = reg_df.copy()
    selected['rank_within_region'] = np.arange(1, len(selected) + 1)
    selection_parts.append(selected)

selected_df = pd.concat(selection_parts, ignore_index=True)

n_hpc_selected = int((selected_df['region'] == 'HPC').sum())
n_acc_selected = int((selected_df['region'] == 'ACC').sum())
if n_hpc_selected == 0 or n_acc_selected == 0:
    raise ValueError(
        f'Expected non-empty electrode sets for both regions; got HPC={n_hpc_selected}, ACC={n_acc_selected}.'
    )

selected_df['selection_metric_name'] = selection_metric
selected_df['selection_metric_value'] = pd.to_numeric(selected_df[selection_metric], errors='coerce')

selected_df = selected_df[
    ['region', 'rank_within_region', 'subject', 'channel', 'selection_metric_name', 'selection_metric_value', 'theta_mean', 'theta_median', 'theta_p90', 'theta_p95', 'theta_max', 'n_rows', 'source_csv']
]

selected_df.to_csv(selection_csv, index=False)
missing_df = pd.DataFrame(missing_rows)
if len(missing_df) > 0:
    missing_df.to_csv(missing_csv, index=False)

print('Saved selection CSV:', selection_csv)
if len(missing_df) > 0:
    print('Saved missing-items CSV:', missing_csv)

display(selected_df)


In [ ]:
# ---- Load selected channels from cue_on-tfr.h5 and prepare plotting tensors ----
plot_rows = []
missing_plot_rows = []

for subject, subj_df in selected_df.groupby('subject'):
    tfr_path = ephys_root / subject / 'scratch' / 'TFR' / f'{event}-tfr.h5'
    if not tfr_path.exists():
        for _, row in subj_df.iterrows():
            missing_plot_rows.append({'subject': subject, 'channel': row['channel'], 'region': row['region'], 'reason': 'missing_tfr_file'})
        continue

    tfr_obj = mne.time_frequency.read_tfrs(str(tfr_path))
    tfr = tfr_obj[0] if isinstance(tfr_obj, list) else tfr_obj
    available_channels = set(tfr.ch_names)

    for _, row in subj_df.iterrows():
        channel = str(row['channel'])
        if channel not in available_channels:
            missing_plot_rows.append({'subject': subject, 'channel': channel, 'region': row['region'], 'reason': 'channel_not_in_tfr'})
            continue

        ch_tfr = tfr.copy().pick([channel])
        mean_tfr = np.nanmean(ch_tfr.data, axis=0).squeeze()  # (freq, time)
        freqs = np.asarray(ch_tfr.freqs, dtype=np.float64)
        times = np.asarray(ch_tfr.times, dtype=np.float64)

        fmask = (freqs >= float(freq_plot_range[0])) & (freqs <= float(freq_plot_range[1]))
        tmask = (times >= float(event_window[0])) & (times <= float(event_window[1]))

        if int(fmask.sum()) == 0 or int(tmask.sum()) == 0:
            missing_plot_rows.append({'subject': subject, 'channel': channel, 'region': row['region'], 'reason': 'empty_freq_or_time_mask'})
            continue

        plot_rows.append({
            'region': row['region'],
            'rank_within_region': int(row['rank_within_region']),
            'subject': subject,
            'channel': channel,
            'selection_metric_value': float(row['selection_metric_value']),
            'theta_mean': float(row['theta_mean']),
            'theta_p90': float(row['theta_p90']),
            'theta_p95': float(row['theta_p95']),
            'tfr_2d': mean_tfr[np.ix_(fmask, tmask)],
            'freqs': freqs[fmask],
            'times': times[tmask]
        })

plot_df = pd.DataFrame(plot_rows)
missing_plot_df = pd.DataFrame(missing_plot_rows)

if len(missing_plot_df) > 0:
    print('Missing items while loading TFR/channels:')
    display(missing_plot_df)

# Fail clearly if no plottable channels remain in either region
have_hpc = int((plot_df['region'] == 'HPC').sum()) if len(plot_df) else 0
have_acc = int((plot_df['region'] == 'ACC').sum()) if len(plot_df) else 0
if have_hpc == 0 or have_acc == 0:
    raise ValueError(
        f'No plottable channels for one or both regions. '
        f'Available HPC={have_hpc}, ACC={have_acc}. See missing table above and {missing_csv}.'
    )

# Keep all plottable channels, ordered by rank within region
plot_df = pd.concat([
    plot_df[plot_df['region'] == 'HPC'].sort_values('rank_within_region'),
    plot_df[plot_df['region'] == 'ACC'].sort_values('rank_within_region')
], ignore_index=True)

print('Plottable selection counts:', plot_df['region'].value_counts().to_dict())


In [ ]:
# ---- Plot helpers ----
def _template_freq_ticks(freqs: np.ndarray, freq_res: int = 5):
    tick_idx = np.arange(0, len(freqs))[::freq_res]
    tick_labels = freqs[::freq_res].astype(int)
    return tick_idx, tick_labels

def _template_time_ticks(times: np.ndarray, time_res: int = 250):
    tick_idx = np.arange(0, len(times))[::time_res]
    tick_labels = np.round(times[::time_res], 2)
    return tick_idx, tick_labels

def _plot_region_grid(region_df: pd.DataFrame, region_name: str, out_png: Path, n_cols: int = 5):
    region_df = region_df.sort_values('rank_within_region').reset_index(drop=True)
    n = len(region_df)
    n_rows = int(math.ceil(n / n_cols))

    # Per-electrode scaling (each panel has its own vmin/vmax)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.8 * n_cols, 3.0 * n_rows), sharex=False, sharey=False)
    axes = np.array(axes).reshape(-1)

    im = None
    for ax_idx, ax in enumerate(axes):
        if ax_idx >= n:
            ax.axis('off')
            continue

        row = region_df.iloc[ax_idx]
        tfr_2d = row['tfr_2d']
        freqs = row['freqs']
        times = row['times']

        local_vmax = float(np.nanpercentile(np.abs(tfr_2d), 98))
        local_vmin = -local_vmax

        im = ax.imshow(
            tfr_2d,
            origin='lower',
            aspect='auto',
            cmap=plot_cmap,
            vmin=local_vmin,
            vmax=local_vmax,
            interpolation=plot_interpolation
        )

        ytick_idx, ytick_labels = _template_freq_ticks(freqs, freq_res=freq_res)
        xtick_idx, xtick_labels = _template_time_ticks(times, time_res=time_res)
        ax.set(yticks=ytick_idx, yticklabels=ytick_labels, xticks=xtick_idx, xticklabels=xtick_labels)

        cue_idx = int(np.argmin(np.abs(times - 0.0)))
        theta_low_idx = int(np.argmin(np.abs(freqs - theta_band[0])))
        theta_high_idx = int(np.argmin(np.abs(freqs - theta_band[1])))
        ax.axvline(cue_idx, color='k', linestyle='--', linewidth=1.0)
        ax.axhline(theta_low_idx, color='k', linestyle=':', linewidth=1.0)
        ax.axhline(theta_high_idx, color='k', linestyle=':', linewidth=1.0)

        ax.set_title(
            f"#{int(row['rank_within_region'])} {row['subject']} {row['channel']}\n"
            f"{selection_metric}={row['selection_metric_value']:.3f}",
            fontsize=9
        )
        ax.set_xlabel('Time from cue onset (s)')
        ax.set_ylabel('Frequency (Hz)')

    fig.suptitle(f'{region_name} cue_on trial-mean TFRs ({n} channels)', fontsize=14)
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    if im is not None:
        fig.colorbar(im, ax=axes.tolist(), shrink=0.7, label='Power (a.u.)')
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', out_png)

def _plot_per_channel_pngs(region_df: pd.DataFrame, region_name: str):
    for _, row in region_df.sort_values('rank_within_region').iterrows():
        fig, ax = plt.subplots(1, 1, figsize=(6, 4))
        tfr_2d = row['tfr_2d']
        freqs = row['freqs']
        times = row['times']
        vmax = float(np.nanpercentile(np.abs(tfr_2d), 98))
        im = ax.imshow(
            tfr_2d,
            origin='lower',
            aspect='auto',
            cmap=plot_cmap,
            vmin=-vmax,
            vmax=vmax,
            interpolation=plot_interpolation
        )
        ytick_idx, ytick_labels = _template_freq_ticks(freqs, freq_res=freq_res)
        xtick_idx, xtick_labels = _template_time_ticks(times, time_res=time_res)
        ax.set(yticks=ytick_idx, yticklabels=ytick_labels, xticks=xtick_idx, xticklabels=xtick_labels)

        cue_idx = int(np.argmin(np.abs(times - 0.0)))
        theta_low_idx = int(np.argmin(np.abs(freqs - theta_band[0])))
        theta_high_idx = int(np.argmin(np.abs(freqs - theta_band[1])))
        ax.axvline(cue_idx, color='k', linestyle='--', linewidth=1.0)
        ax.axhline(theta_low_idx, color='k', linestyle=':', linewidth=1.0)
        ax.axhline(theta_high_idx, color='k', linestyle=':', linewidth=1.0)
        ax.set_title(
            f"{region_name} #{int(row['rank_within_region'])} {row['subject']} {row['channel']} | "
            f"{selection_metric}={row['selection_metric_value']:.3f}"
        )
        ax.set_xlabel('Time from cue onset (s)')
        ax.set_ylabel('Frequency (Hz)')
        fig.colorbar(im, ax=ax, label='Power (a.u.)')
        fig.tight_layout()

        out_file = channel_figure_root / f"{region_name.lower()}_{int(row['rank_within_region']):02d}_{row['subject']}_{row['channel']}.png"
        fig.savefig(out_file, dpi=300, bbox_inches='tight')
        plt.close(fig)


In [ ]:
channel_figure_root

In [ ]:
# ---- Render HPC / ACC grids + optional per-channel PNGs ----
assert plot_cmap == 'viridis'
assert plot_interpolation == 'bicubic'
assert np.array_equal(np.arange(0, 30)[::5], np.arange(0, 30)[::freq_res])
hpc_df = plot_df[plot_df['region'] == 'HPC'].copy()
acc_df = plot_df[plot_df['region'] == 'ACC'].copy()

_plot_region_grid(hpc_df, 'HPC', hpc_grid_png, n_cols=5)
_plot_region_grid(acc_df, 'ACC', acc_grid_png, n_cols=5)

_plot_per_channel_pngs(hpc_df, 'HPC')
_plot_per_channel_pngs(acc_df, 'ACC')

print('Per-channel PNG directory:', channel_figure_root)
